# Projet Challenge 2 – Modules 4 et 5 : évaluation, seuil par le coût, explications

Part des sorties du notebook 03 (dans le Drive) : `variables_v2.parquet`, `modeles_v1.joblib`, `config_v1.json`.

- **Module 4** : budget d'alertes *k* justifié, métriques de classement sur validation **et** test, courbes PR,
  seuil τ choisi sur la **validation** en minimisant le coût, puis **confirmé une seule fois sur le test** (F2, MCC).
- **Module 5** : SHAP global (beeswarm), explication d'alertes (waterfall + codes motifs), analyse des erreurs.
- **Export** : dossier `artefacts/` pour l'application Streamlit et l'API FastAPI.

## 0. Chargement

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, shap, xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DOSSIER = "/content/drive/MyDrive/fraude_tabformer"
final = pd.read_parquet(f"{DOSSIER}/variables_v2.parquet")
config = json.load(open(f"{DOSSIER}/config_v1.json"))
modeles = joblib.load(f"{DOSSIER}/modeles_v1.joblib")
VARIABLES, CATEGORIELLE = config["VARIABLES"], config["CATEGORIELLE"]
UTILISE_NUM, BETA = bool(config.get("UTILISE_NUM", False)), float(config["BETA"])
print("Stratégie :", config["STRATEGIE"], "| BETA :", BETA, "| matrice numérique :", UTILISE_NUM)
print(final.groupby("partie")["fraude"].agg(transactions="size", fraudes="sum").to_string())

In [ ]:
# Mêmes encodages qu'au notebook 03 (appris sur l'entraînement, lus depuis config_v1.json)
mcc_risque, medianes = pd.Series(config["mcc_risque"]), pd.Series(config["medianes"])

def matrice_xgb(df):
    X = df[VARIABLES].copy()
    X[CATEGORIELLE] = pd.Categorical(df[CATEGORIELLE], categories=config["categories_mcc"])
    return X

def matrice_num(df):
    X = df[VARIABLES].fillna(medianes).copy()
    X["mcc_risque"] = df[CATEGORIELLE].map(mcc_risque).fillna(config["taux_global"]).astype("float32")
    return X

def corriger_proba(ps, beta):
    return beta * ps / (beta * ps - ps + 1)

def scorer_tous(df):
    Xn = matrice_num(df)
    Xb = Xn if UTILISE_NUM else matrice_xgb(df)
    s = {"XGBoost": modeles["xgb"].predict_proba(Xb)[:, 1]}
    if BETA != 1:
        s["XGBoost"] = corriger_proba(s["XGBoost"], BETA)
    s["Régression logistique"] = modeles["lr"].predict_proba(Xn)[:, 1]
    s["Isolation Forest"] = -modeles["iforest"].score_samples(Xn)
    Z = modeles["sc_ae"].transform(Xn)
    s["Autoencodeur"] = ((np.clip(Z, -10, 10) - modeles["ae"].predict(Z)) ** 2).mean(axis=1)
    return pd.DataFrame(s, index=df.index).astype("float32")

va = final[final["partie"] == "validation"].reset_index(drop=True)
te = final[final["partie"] == "test"].reset_index(drop=True)
t0 = time.time()
S_va, S_te = scorer_tous(va), scorer_tous(te)
y_va, y_te = va["fraude"].to_numpy(), te["fraude"].to_numpy()
m_va, m_te = va["montant"].to_numpy(), te["montant"].to_numpy()
print(f"Scores calculés en {time.time() - t0:.0f} s")

---
# Module 4 – Évaluation et seuil

## 4.1 Budget d'alertes *k*

Hypothèse à défendre : **une analyste traite environ 10 dossiers par jour** pour ce portefeuille de 500 clients
(environ 1 400 transactions par jour). *k* = budget × nombre de jours de la période. On regarde aussi la
sensibilité au budget.

In [ ]:
BUDGET_JOUR = int(config.get("BUDGET_JOUR", 10))
jours_va, jours_te = va["date"].dt.normalize().nunique(), te["date"].dt.normalize().nunique()
K_VA, K_TE = BUDGET_JOUR * jours_va, BUDGET_JOUR * jours_te
print(f"Transactions par jour : {len(va) / jours_va:,.0f} (validation), {len(te) / jours_te:,.0f} (test)")
print(f"Fraudes par jour : {y_va.sum() / jours_va:.2f} (validation), {y_te.sum() / jours_te:.2f} (test)")
print(f"k validation = {K_VA:,} | k test = {K_TE:,}  ({BUDGET_JOUR} alertes par jour)")

def prk(y, s, k):
    o = np.argsort(-s, kind="stable")[:k]; vp = y[o].sum()
    return vp / k, vp / y.sum()

sens = []
for b in [2, 5, 10, 20, 50]:
    p, r = prk(y_va, S_va["XGBoost"].to_numpy(), b * jours_va)
    sens.append({"alertes / jour": b, "k": b * jours_va, "précision@k": p, "rappel@k": r})
sens = pd.DataFrame(sens); print("\nXGBoost en validation selon le budget :"); print(sens.round(3).to_string(index=False))

## 4.2 Métriques de classement : validation et test

In [ ]:
def tableau_classement(S, y, k):
    lignes = []
    for nom in S.columns:
        s = S[nom].to_numpy(); p, r = prk(y, s, k)
        lignes.append({"modèle": nom, "PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s),
                       "précision@k": p, "rappel@k": r})
    return pd.DataFrame(lignes).set_index("modèle")

classement = pd.concat({"validation": tableau_classement(S_va, y_va, K_VA),
                        "test": tableau_classement(S_te, y_te, K_TE)}, axis=1)
print(f"Hasard : PR-AUC = {y_va.mean():.4f} (validation), {y_te.mean():.4f} (test)")
display(classement.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (nom_p, S, y) in zip(axes, [("validation 2018", S_va, y_va), ("test 2019", S_te, y_te)]):
    for nom in S.columns:
        p, r, _ = precision_recall_curve(y, S[nom])
        ax.plot(r, p, label=f"{nom} ({average_precision_score(y, S[nom]):.3f})")
    ax.axhline(y.mean(), color="grey", ls=":", label="hasard")
    ax.set_title(f"Précision-rappel – {nom_p}"); ax.set_xlabel("rappel")
axes[0].set_ylabel("précision"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4.3 Seuil par le coût (validation)

Coût(τ) = Σ montants des fraudes manquées + C_a × nombre d'alertes.
**C_a = 5 dollars** : environ 10 minutes d'analyste à 30 dollars de l'heure (hypothèse, avec analyse de sensibilité).
Les montants négatifs ne comptent pas comme perte.

In [ ]:
COUT_ALERTE = 5.0

def metriques_seuil(y, s, m, tau, ca=COUT_ALERTE):
    a = s >= tau
    vp, fp = int((a & (y == 1)).sum()), int((a & (y == 0)).sum())
    fn, vn = int((~a & (y == 1)).sum()), int((~a & (y == 0)).sum())
    pr = vp / (vp + fp) if vp + fp else 0.0; rc = vp / (vp + fn)
    f2 = 5 * pr * rc / (4 * pr + rc) if pr + rc else 0.0
    d = np.sqrt(float(vp + fp) * (vp + fn) * (vn + fp) * (vn + fn)); mcc = (vp * vn - fp * fn) / d if d else 0.0
    mp = np.clip(m, 0, None); perte = mp[(~a) & (y == 1)].sum()
    return {"seuil": tau, "alertes": vp + fp, "alertes / jour": None, "VP": vp, "FP": fp, "FN": fn,
            "précision": pr, "rappel": rc, "F2": f2, "MCC": mcc, "fraudes manquées ($)": perte,
            "coût d'analyse ($)": ca * (vp + fp), "coût total ($)": perte + ca * (vp + fp),
            "sans modèle ($)": mp[y == 1].sum()}

def courbe_cout(y, s, m, ca=COUT_ALERTE):
    o = np.argsort(-s, kind="stable"); ss, yy, mm = s[o], y[o], np.clip(m[o], 0, None)
    n = np.arange(0, len(s) + 1)
    det = np.concatenate([[0], np.cumsum(mm * yy)]); vp = np.concatenate([[0], np.cumsum(yy)])
    cout = (mm[yy == 1].sum() - det) + ca * n
    seuils = np.concatenate([[np.inf], ss])
    return pd.DataFrame({"alertes": n, "seuil": seuils, "VP": vp, "coût total ($)": cout})

s_va_xgb, s_te_xgb = S_va["XGBoost"].to_numpy(), S_te["XGBoost"].to_numpy()
cc = courbe_cout(y_va, s_va_xgb, m_va)
i = int(cc["coût total ($)"].idxmin())
TAU = float(cc.loc[i, "seuil"])
print(f"Seuil retenu τ = {TAU:.4f} -> {int(cc.loc[i, 'alertes']):,} alertes en validation "
      f"({cc.loc[i, 'alertes'] / jours_va:.1f} par jour)")
print(f"Coût minimal : {cc.loc[i, 'coût total ($)']:,.0f} $ contre {np.clip(m_va, 0, None)[y_va == 1].sum():,.0f} $ sans modèle")

fig, ax = plt.subplots(figsize=(9, 3.8))
lim = int(min(len(cc) - 1, max(4 * cc.loc[i, "alertes"], 3 * K_VA)))
ax.plot(cc["alertes"][:lim], cc["coût total ($)"][:lim], color="crimson")
ax.axvline(cc.loc[i, "alertes"], ls="--", color="grey"); ax.axvline(K_VA, ls=":", color="steelblue")
ax.text(cc.loc[i, "alertes"], ax.get_ylim()[1] * 0.95, " coût minimal", fontsize=8)
ax.text(K_VA, ax.get_ylim()[1] * 0.85, f" budget {BUDGET_JOUR}/jour", fontsize=8, color="steelblue")
ax.set_xlabel("nombre d'alertes (validation)"); ax.set_ylabel("coût total ($)"); ax.set_title("Coût selon le nombre d'alertes – XGBoost")
plt.tight_layout(); plt.show()

In [ ]:
# Sensibilité du seuil au coût d'une alerte
sens_cout = []
for ca in [1, 2, 5, 10, 20, 50]:
    c = courbe_cout(y_va, s_va_xgb, m_va, ca); j = int(c["coût total ($)"].idxmin())
    sens_cout.append({"C_a ($)": ca, "seuil": c.loc[j, "seuil"], "alertes / jour": c.loc[j, "alertes"] / jours_va,
                      "rappel": c.loc[j, "VP"] / y_va.sum(), "coût ($)": c.loc[j, "coût total ($)"]})
sens_cout = pd.DataFrame(sens_cout); print(sens_cout.round(4).to_string(index=False))

## 4.4 Confirmation sur le test (une seule fois)

Le seuil τ fixé en validation est appliqué tel quel en 2019. On compare au seuil qui aurait été optimal
sur le test (inconnu en pratique) pour mesurer la perte due au transfert.

In [ ]:
r_va = metriques_seuil(y_va, s_va_xgb, m_va, TAU); r_va["alertes / jour"] = r_va["alertes"] / jours_va
r_te = metriques_seuil(y_te, s_te_xgb, m_te, TAU); r_te["alertes / jour"] = r_te["alertes"] / jours_te
ct = courbe_cout(y_te, s_te_xgb, m_te); j = int(ct["coût total ($)"].idxmin())
r_or = metriques_seuil(y_te, s_te_xgb, m_te, float(ct.loc[j, "seuil"])); r_or["alertes / jour"] = r_or["alertes"] / jours_te
au_seuil = pd.DataFrame({"validation (τ choisi)": r_va, "test (même τ)": r_te, "test (τ oracle)": r_or}).T
display(au_seuil.round(4))
gain = 1 - r_te["coût total ($)"] / r_te["sans modèle ($)"]
print(f"Sur le test, le modèle réduit le coût de la fraude de {gain * 100:.1f} % par rapport à l'absence de modèle.")

In [ ]:
# Même procédure pour tous les modèles : seuil choisi en validation, appliqué au test
lignes = []
for nom in S_va.columns:
    c = courbe_cout(y_va, S_va[nom].to_numpy(), m_va); t = float(c.loc[int(c["coût total ($)"].idxmin()), "seuil"])
    r = metriques_seuil(y_te, S_te[nom].to_numpy(), m_te, t)
    lignes.append({"modèle": nom, **{k: r[k] for k in ["alertes", "précision", "rappel", "F2", "MCC", "coût total ($)"]}})
seuil_tous = pd.DataFrame(lignes).set_index("modèle")
print("Test 2019, seuil de chaque modèle choisi en validation :"); display(seuil_tous.round(4))

### Calibration des probabilités

Au Module 3, la probabilité moyenne prédite (0,00034) était 4 fois plus faible que le taux réel de 2018 (0,00134).
On vérifie par déciles de score : un modèle bien calibré a des points sur la diagonale.

In [ ]:
for nom_p, y, s in [("validation", y_va, s_va_xgb), ("test", y_te, s_te_xgb)]:
    print(f"{nom_p:10s} probabilité moyenne prédite {s.mean():.5f} | taux réel {y.mean():.5f} | rapport {y.mean() / s.mean():.1f}")
d = pd.DataFrame({"s": s_te_xgb, "y": y_te})
d["groupe"] = pd.qcut(d["s"].rank(method="first"), 20, labels=False)
cal = d.groupby("groupe").agg(predit=("s", "mean"), observe=("y", "mean"))
fig, ax = plt.subplots(figsize=(5, 4))
ax.loglog(cal["predit"].clip(1e-7), cal["observe"].clip(1e-7), "o-", color="crimson")
lim = [1e-7, 1]; ax.plot(lim, lim, ":", color="grey"); ax.set_xlabel("probabilité prédite"); ax.set_ylabel("taux observé")
ax.set_title("Calibration – test (20 groupes)"); plt.tight_layout(); plt.show()

## 4.5 Pour aller plus loin : décision selon le montant

Un seuil unique traite pareil une fraude de 5 dollars et une fraude de 2 000 dollars. La règle « alerter si p × montant > C_a »
minimise le coût **espéré** transaction par transaction (si les probabilités sont bien calibrées).

In [ ]:
for nom_p, y, s, m in [("validation", y_va, s_va_xgb, m_va), ("test", y_te, s_te_xgb, m_te)]:
    a = s * np.clip(m, 0, None) > COUT_ALERTE
    mp = np.clip(m, 0, None); cout = mp[(~a) & (y == 1)].sum() + COUT_ALERTE * a.sum()
    print(f"{nom_p:10s} règle p × montant : {a.sum():6,} alertes, rappel {(a & (y == 1)).sum() / y.sum():.3f}, coût {cout:,.0f} $")
print(f"Rappel – seuil unique : validation {r_va['coût total ($)']:,.0f} $, test {r_te['coût total ($)']:,.0f} $")

---
# Module 5 – Explainable AI

Contributions SHAP exactes calculées par XGBoost (`pred_contribs=True`, en log-odds). Les deux variables d'heure
(sin, cos) sont regroupées en « heure ».

In [ ]:
booster = modeles["xgb"].get_booster()

def contributions(df):
    X = matrice_num(df) if UTILISE_NUM else matrice_xgb(df)
    c = booster.predict(xgb.DMatrix(X, enable_categorical=True), pred_contribs=True)
    C = pd.DataFrame(c[:, :-1], columns=X.columns, index=df.index)
    C.insert(0, "heure", C.pop("heure_sin") + C.pop("heure_cos"))
    valeurs = df[[col for col in X.columns if col not in ("heure_sin", "heure_cos", CATEGORIELLE, "mcc_risque")]].copy()
    valeurs.insert(0, "heure", (np.arctan2(df["heure_sin"], df["heure_cos"]) / (2 * np.pi) * 24) % 24)
    nom_mcc = "mcc_risque" if UTILISE_NUM else CATEGORIELLE
    valeurs[nom_mcc] = df[CATEGORIELLE].map(mcc_risque).fillna(config["taux_global"]).to_numpy()  # couleur = risque du groupe
    return C, valeurs[C.columns].astype(float), c[:, -1]

ech_shap = te.sample(5000, random_state=0)
C, V, base = contributions(ech_shap)
expl = shap.Explanation(values=C.to_numpy(), base_values=base, data=V.to_numpy(), feature_names=list(C.columns))
shap.plots.beeswarm(expl, max_display=15, show=False); plt.title("SHAP global – test 2019 (5 000 transactions)"); plt.show()
imp = C.abs().mean().sort_values(ascending=False)
print((imp / imp.sum() * 100).round(1).head(12).to_string())

### 5.2 Explication d'alertes : waterfall et codes motifs

In [ ]:
LIB = {"heure": "heure", "etranger": "à l'étranger", "en_ligne": "paiement en ligne", "nouveau_marchand": "nouveau marchand",
       "ratio_montant_30j": "montant / moyenne 30 j", "montant": "montant", "log_montant": "montant (log)",
       "nb_etranger_24h": "transactions à l'étranger sur 24 h", "nb_nouveaux_marchands_24h": "nouveaux marchands sur 24 h",
       "montant_24h": "montant sur 24 h", "nb_tx_1h": "transactions sur 1 h", "nb_tx_24h": "transactions sur 24 h",
       "h_depuis_prec": "temps depuis la précédente", "mcc_groupe": "catégorie de marchand", "mcc_risque": "catégorie de marchand",
       "nouvelle_ville": "nouvelle ville", "err_cvv": "erreur CVV", "une_erreur": "erreur à l'autorisation"}

def motifs(c, ligne, n=5):
    pos = c[c > 0.05].sort_values(ascending=False).head(n)
    out = []
    for var, val in pos.items():
        v = ligne.get(var, np.nan)
        if var == "ratio_montant_30j": txt = f"montant = {v:.1f} × la moyenne des 30 derniers jours"
        elif var in ("montant", "log_montant"): txt = f"montant de {ligne['montant']:,.0f} $"
        elif var in ("mcc_groupe", "mcc_risque"): txt = f"catégorie de marchand {ligne[CATEGORIELLE]}"
        elif var == "heure": txt = f"heure : {(np.arctan2(ligne['heure_sin'], ligne['heure_cos']) / (2*np.pi) * 24) % 24:.1f} h"
        elif var == "h_depuis_prec": txt = f"{v:.2f} h depuis la transaction précédente"
        else: txt = f"{LIB.get(var, var)} = {v:g}"
        out.append(f"+{val:.2f}  {txt}")
    return out

def expliquer(df_lignes, titre):
    C1, V1, b1 = contributions(df_lignes)
    for k in range(len(df_lignes)):
        ligne = df_lignes.iloc[k]
        print(f"\n=== {titre} | score {S_te.loc[df_lignes.index[k], 'XGBoost']:.3f} | fraude réelle : {int(ligne['fraude'])} "
              f"| {ligne['date']} | {ligne['montant']:.2f} $")
        print("Codes motifs :"); [print("  ", m) for m in motifs(C1.iloc[k], ligne)]
        shap.plots.waterfall(shap.Explanation(values=C1.iloc[k].to_numpy(), base_values=b1[k], data=V1.iloc[k].to_numpy(),
                                              feature_names=list(C1.columns)), max_display=10, show=False)
        plt.title(titre); plt.show()

alerte = te["fraude"].eq(1) & (s_te_xgb >= TAU)
fausse = te["fraude"].eq(0) & (s_te_xgb >= TAU)
manquee = te["fraude"].eq(1) & (s_te_xgb < TAU)
expliquer(te[alerte].loc[[S_te.loc[alerte, "XGBoost"].idxmax()]], "Vraie alerte la plus sûre")
if fausse.any(): expliquer(te[fausse].loc[[S_te.loc[fausse, "XGBoost"].idxmax()]], "Fausse alerte la plus sûre")
if manquee.any(): expliquer(te[manquee].loc[[S_te.loc[manquee, "XGBoost"].idxmin()]], "Fraude manquée la moins détectée")

### 5.3 Quels motifs reviennent dans les alertes ?

In [ ]:
Ca, _, _ = contributions(te[s_te_xgb >= TAU])
premier = Ca.idxmax(axis=1).map(lambda v: LIB.get(v, v)).value_counts(normalize=True).mul(100).round(1)
print("Motif principal des alertes du test (%) :"); print(premier.head(10).to_string())
print("\nFraudes manquées : canal et lieu")
fm = te[manquee]
print(f"  en ligne {fm['en_ligne'].mean() * 100:.0f} % | à l'étranger {fm['etranger'].mean() * 100:.0f} % | "
      f"montant médian {fm['montant'].median():.0f} $ (vs {te.loc[alerte, 'montant'].median():.0f} $ pour les fraudes détectées)")

---
# Export pour l'application et l'API

Crée `artefacts/` dans le Drive. Télécharge ensuite ce dossier et place-le à la racine du dépôt GitHub.

In [ ]:
ART = f"{DOSSIER}/artefacts"; os.makedirs(ART, exist_ok=True)

# 1. Modèle (format JSON portable) et configuration
booster.save_model(f"{ART}/modele_xgb.json")
def propre(d): return {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in d.items()}
cfg = {**config, "modele": "XGBoost", "seuil": TAU, "cout_alerte": COUT_ALERTE, "BUDGET_JOUR": BUDGET_JOUR,
       "UTILISE_NUM": UTILISE_NUM, "periode": {"entraînement": "2015-2017", "validation": "2018", "test": "2019"},
       "metriques_validation": propre(r_va), "metriques_test": propre(r_te), "version_donnees": "variables_v2",
       "jours": {"validation": int(jours_va), "test": int(jours_te)}}
json.dump(cfg, open(f"{ART}/config.json", "w"), ensure_ascii=False, indent=1, default=float)

# 2. Scores de tous les modèles (validation + test) pour les pages Modèles et Seuil
scores = pd.concat([
    pd.DataFrame({"date": va["date"], "partie": "validation", "fraude": y_va.astype("int8"), "montant": m_va.astype("float32")}).join(S_va),
    pd.DataFrame({"date": te["date"], "partie": "test", "fraude": y_te.astype("int8"), "montant": m_te.astype("float32")}).join(S_te),
], ignore_index=True)
scores["partie"] = scores["partie"].astype("category")
scores.to_parquet(f"{ART}/scores.parquet", index=False, compression="zstd")

# 3. Transactions détaillées pour la page Explications : alertes les plus fortes, toutes les fraudes, échantillon aléatoire
def selection(df, s, nom):
    df = df.assign(score=s, partie=nom)
    top = df.nlargest(3000, "score").assign(groupe="alertes les plus fortes")
    fr = df[df["fraude"] == 1].assign(groupe="fraudes")
    al = df.sample(3000, random_state=0).assign(groupe="échantillon aléatoire")
    return pd.concat([top, fr, al]).drop_duplicates(subset=["date", "carte", "montant", "groupe"])
transactions = pd.concat([selection(va, s_va_xgb, "validation"), selection(te, s_te_xgb, "test")], ignore_index=True)
transactions.drop(columns=["Merchant Name"]).to_parquet(f"{ART}/transactions_app.parquet", index=False, compression="zstd")

# 4. Agrégats pour la page Données (Module 1)
def agr(cle):
    t = final.groupby(cle, observed=True)["fraude"].agg(transactions="size", fraudes="sum").reset_index()
    t["taux"] = t["fraudes"] / t["transactions"]; return t
lieu = np.select([final["en_ligne"] == 1, final["etranger"] == 1], ["en ligne", "étranger"], "États-Unis (magasin)")
canal = np.select([final["en_ligne"] == 1, final["bande"] == 1], ["en ligne", "bande magnétique"], "puce")
tranche = pd.cut(final["montant"], [-np.inf, 0, 10, 50, 100, 250, 500, 1000, np.inf],
                 labels=["≤0", "0-10", "10-50", "50-100", "100-250", "250-500", "500-1000", ">1000"]).astype(str)
eda = {
    "resume": {"transactions": int(len(final)), "fraudes": int(final["fraude"].sum()), "taux": float(final["fraude"].mean()),
               "montant_fraude": float(final.loc[final["fraude"] == 1, "montant"].clip(lower=0).sum()),
               "cartes": int(final["carte"].nunique()), "debut": str(final["date"].min()), "fin": str(final["date"].max())},
    "partie": agr("partie").to_dict("records"),
    "mois": agr(final["date"].dt.to_period("M").astype(str).rename("mois")).to_dict("records"),
    "heure": agr(final["date"].dt.hour.rename("heure")).to_dict("records"),
    "jour": agr(final["date"].dt.dayofweek.rename("jour")).to_dict("records"),
    "canal": agr(pd.Series(canal, name="canal")).to_dict("records"),
    "lieu": agr(pd.Series(lieu, name="lieu")).to_dict("records"),
    "tranche_montant": agr(tranche.rename("tranche")).to_dict("records"),
    "mcc": agr(final[CATEGORIELLE].rename("mcc")).query("transactions >= 200").sort_values("taux", ascending=False).head(25).to_dict("records"),
    "erreurs": [{"erreur": c, "transactions": int(final[c].sum()), "fraudes": int(final.loc[final[c] == 1, "fraude"].sum())}
                for c in [c for c in VARIABLES if c.startswith("err_")]],
    "hist_montant": {str(k): np.histogram(np.log10(final.loc[(final["fraude"] == k) & (final["montant"] > 0), "montant"]),
                                          bins=60, range=(-1, 4), density=True)[0].round(5).tolist() for k in (0, 1)},
}
json.dump(eda, open(f"{ART}/eda.json", "w"), ensure_ascii=False, default=float)

# 5. Tableaux de résultats
classement.to_csv(f"{ART}/classement.csv"); au_seuil.to_csv(f"{ART}/au_seuil.csv"); seuil_tous.to_csv(f"{ART}/seuil_tous.csv")
sens_cout.to_csv(f"{ART}/sensibilite_cout.csv", index=False); sens.to_csv(f"{ART}/sensibilite_budget.csv", index=False)

for f in sorted(os.listdir(ART)):
    print(f"{f:28s} {os.path.getsize(f'{ART}/{f}') / 1e6:7.2f} Mo")

### Exemple pour l'API : une vraie alerte du test avec l'historique brut de sa carte

In [ ]:
ech = pd.read_parquet(f"{DOSSIER}/echantillon_500_users.parquet")
cible = te[alerte].loc[S_te.loc[alerte, "XGBoost"].idxmax()]
u, k = map(int, cible["carte"].split("_"))
brut = ech[(ech["User"] == u) & (ech["Card"] == k)].sort_values("date")
# Historique COMPLET de la carte : les variables de nouveauté et d'habitude dépendent de tout le passé
hist = brut[brut["date"] < cible["date"]]
tx = brut[brut["date"] == cible["date"]].iloc[[0]]

def vers_api(r):
    est_na = lambda v: v is None or (isinstance(v, float) and np.isnan(v)) or pd.isna(v)
    return {"date": r["date"].isoformat(), "montant": float(r["Amount"]), "canal": str(r["Use Chip"]),
            "marchand": str(r["Merchant Name"]), "ville": None if est_na(r["Merchant City"]) else str(r["Merchant City"]),
            "etat": None if est_na(r["Merchant State"]) else str(r["Merchant State"]), "mcc": int(r["MCC"]),
            "erreurs": None if est_na(r["Errors?"]) else str(r["Errors?"])}
exemple = {"carte": cible["carte"], "historique": [vers_api(r) for _, r in hist.iterrows()], "transaction": vers_api(tx.iloc[0])}
json.dump({"demande": exemple, "score_attendu": float(S_te.loc[cible.name, "XGBoost"])},
          open(f"{ART}/exemple_api.json", "w"), ensure_ascii=False, indent=1)
print(f"Exemple : carte {cible['carte']}, {len(hist)} transactions d'historique, score attendu {S_te.loc[cible.name, 'XGBoost']:.4f}")

In [ ]:
# Télécharger le dossier artefacts en un seul zip
import shutil
from google.colab import files
shutil.make_archive("/content/artefacts", "zip", ART)
files.download("/content/artefacts.zip")

## À rapporter pour l'analyse

Sorties de **4.1, 4.2, 4.3 (seuil et sensibilité), 4.4, 4.5, 5.1 (importance), 5.2 (codes motifs), 5.3**, plus les
captures du beeswarm et des waterfalls.